In [1]:
# 공통 설정
# .env 파일에서 API Key와 기본 모델 명을 읽어온다
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY를 환경변수로 설정하세요")

client = OpenAI(api_key=api_key)
DEFAULT_MODEL = os.getenv("OPENAI_DEFAULT_MODEL","gpt-4.1-mini")

print("OpenAI client 준비 완료")
print("기본 모델 :", DEFAULT_MODEL)

OpenAI client 준비 완료
기본 모델 : gpt-4.1-mini


# streaming
일반 응답 방식은 모델이 전체 답변을 생성한 뒤 한번에 결과를 돌려준다.
코드가 단순하고 후처리가 쉽지만, 답변이 긴 경우 사용자는 기다리는 시간이 길게 느껴질 수 있다.

In [2]:
import time

start = time.perf_counter()
response = client.responses.create(
    model=DEFAULT_MODEL,
    instructions="너는 초급 개발자에게 쉽게 설명하는 AI 강사이다",
    input="Streaming 응답이 필요한 이유를 설명해줘"
)

elapsed = time.perf_counter() - start

print(response.output_text)
print(f"elapsed : {elapsed:.2f}s")

물론이야! 쉽게 설명해 줄게.

**Streaming 응답이 필요한 이유**는 다음과 같아:

1. **빠르게 결과를 보여주기 위해서**  
   일반적으로 서버가 데이터를 다 준비한 다음 한꺼번에 보내주면, 사용자는 완성된 결과를 받기 전까지 기다려야 해.  
   하지만 스트리밍 응답은 서버가 데이터를 조금씩 준비하는 대로 바로바로 보내줘서, 사용자는 기다리지 않고도 바로바로 내용을 확인할 수 있어.

2. **실시간 업데이트가 필요할 때**  
   예를 들어 채팅, 라이브 방송 자막, 실시간 데이터 모니터링 같은 상황에서는 데이터가 계속 바뀌고 추가되어야 해.  
   이런 경우 스트리밍을 사용하면 새로운 정보를 곧바로 받을 수 있어, 더 자연스럽고 빠르게 반응할 수 있지.

3. **서버와 클라이언트 부담 완화**  
   응답을 한꺼번에 보내면 서버가 긴 시간 동안 데이터를 모두 모아야 하고, 클라이언트도 큰 데이터를 한 번에 받아서 처리해야 해.  
   스트리밍은 데이터를 조금씩 나눠 보내주니, 부담이 나눠지고 메모리와 네트워크 사용을 효율적으로 할 수 있어.

쉽게 말해, **"결과가 완성될 때까지 기다릴 필요 없이, 계속 부분부분 받아보면서 빠르게 볼 수 있게 하는 방법"** 이 바로 스트리밍 응답이야!  

궁금한 점 있으면 더 물어봐~
elapsed : 8.62s


In [3]:
# streaming 응답 방식 적용
stream = client.responses.create(
    model=DEFAULT_MODEL,
    instructions="너는 초급 개발자에게 쉽게 설명하는 AI 강사이다",
    input="Streaming 응답을 식당 주문 처리에 비유해서 설명해줘",
    stream=True
)

for event in stream:
    # delta 이벤트는 새로 생성된 텍스트 조각을 의미한다.
    if event.type == 'response.output_text.delta':
        print(event.delta, end="",flush=True)

좋아요! Streaming 응답을 식당 주문 처리에 비유해서 쉽게 설명해볼게요.

---

### 1. **일반적인 응답 처리 (한 번에 모두 제공) - 한꺼번에 음식 전부 서빙하기**

- 식당에서 손님이 주문을 하면,
- 주방에서 모든 음식을 다 만들고,
- 다 완성된 후에 한 번에 손님에게 음식을 전부 서빙해요.

이게 일반적인 응답 처리 방식이에요. 즉, 서버가 **모든 데이터를 다 준비해서 한꺼번에 보내는 것**이에요.

---

### 2. **Streaming 응답 처리 - 조금씩 서빙하기**

- 이번에는 손님이 여러 가지 음식을 주문했어요.
- 주방에서 먼저 국을 만들면 바로 손님에게 국을 가져다주고,
- 이어서 밥, 반찬 등 하나씩 완성되는 대로 빠르게 가져다줘요.

즉, 음식을 전부 다 만들 때까지 기다리지 않고, **준비된 음식부터 즉시 나눠주는 것**이에요.

---

### 3. **왜 Streaming 응답을 쓸까?**

- 손님은 음식을 빨리 받으면서 식사를 시작할 수 있고,
- 기다리는 시간을 줄여서 만족도가 높아져요.
- 서버도 데이터를 조금씩 보내면서 처리할 수 있기 때문에 부담을 줄일 수 있어요.

---

### 요약

| 상황                      | 식당 비유                           | 서버 응답 비유              |
|-------------------------|---------------------------------|-------------------------|
| 일반 응답               | 모든 음식 다 준비 후 한꺼번에 서빙       | 모든 데이터 완성 후 한 번에 전송    |
| Streaming 응답          | 음식 하나씩 완성되는 대로 조금씩 서빙       | 데이터 조각조각 나오는 대로 바로 전송 |

---

이해하기 쉬웠나요? 혹시 더 궁금한 점 있으면 알려줘요!

# 토큰 사용량 확인
API 응답 객체에는 입력 토큰, 출력 토큰, 전체 토큰 정보를 담은 'usage'가 포함된다.

In [4]:
response = client.responses.create(
    model=DEFAULT_MODEL,
    instructions="너는 간결하게 답하는 AI 강사이다",
    input="토큰이 무엇인지 설명해줘",
)

print(response.output_text)
print()
print(response.usage)

토큰은 텍스트를 처리할 때 의미 있는 작은 단위입니다. 예를 들어, 단어, 문장 부호, 혹은 단어의 일부가 될 수 있어요. 자연어처리에서는 텍스트를 토큰 단위로 나누어 분석하고 처리합니다.

ResponseUsage(input_tokens=30, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=64, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=94)


## 비용 추정 함수

In [5]:
PRICE_PER_1M_INPUT = 0.40
PRICE_PER_1M_OUTPUT = 1.60

def estimate_cost(usage):
    """
    responses api usage 객체를 받아 예상 비용을 계산하는 함수
    """
    if usage is None:
        return None
    
    input_tokens = getattr(usage, 'input_tokens',0) or 0
    output_tokens = getattr(usage, 'output_tokens',0) or 0
    
    return (input_tokens / 1_000_000) * PRICE_PER_1M_INPUT + (output_tokens / 1_000_000) * PRICE_PER_1M_OUTPUT

cost = estimate_cost(response.usage)
print("estimate cost($) : ",cost)

estimate cost($) :  0.00011439999999999999


## api 호출 로그 남기기
실험을 많이 할 수록 어떤 프롬프트가 좋았는지 기억하기 어렵다.
따라서 프롬프트, 모델명, 파라미터, 응답 시간, 토큰 사용량, 비용, 출력 결과물등을 표로 남긴다. 

In [6]:
import pandas as pd

logs = []

def logged_response(prompt,instructions,model=DEFAULT_MODEL,temperature=0.3):
    start = time.perf_counter()
    response = client.responses.create(
        model=model,
        instructions=instructions,
        input=prompt,
        temperature=temperature
    )
    elapsed = time.perf_counter()-start
    usage = response.usage

    row = {
        "model":model,
        "temperature":temperature,
        "prompt":prompt,
        "output":response.output_text,
        "elapsed_sec":round(elapsed,3),
        "input_tokens":getattr(usage,"input_tokens",None) if usage else None,
        "output_tokens":getattr(usage,"output_tokens",None) if usage else None,
        "total_tokens":getattr(usage,"totalput_tokens",None) if usage else None,
        "estimate_cost_usd":estimate_cost(usage)
    }
    logs.append(row)
    return response.output_text

In [7]:
prompts = [
    "RAG를 한 문장으로 설명해줘.",
    "RAG를 초급 개발자에게 5문장으로 설명해줘",
    "RAG를 백엔드 개발자의 관점에서 설명해줘"
]

for p in prompts:
    print(logged_response(p,instructions="너는 AI 강사이다.",temperature=0.2))
    print("-"*60)

log_df = pd.DataFrame(logs)
log_df

RAG( Retrieval-Augmented Generation )는 외부 지식베이스에서 정보를 검색해 이를 바탕으로 더 정확하고 풍부한 텍스트를 생성하는 AI 모델입니다.
------------------------------------------------------------
RAG는 "Retrieval-Augmented Generation"의 약자입니다. 먼저, 관련된 정보를 데이터베이스나 문서에서 찾아오고(Retrieval), 그 정보를 바탕으로 답변을 생성(Generation)하는 방식입니다. 이렇게 하면 AI가 더 정확하고 풍부한 답변을 만들 수 있어요. 예를 들어, 챗봇이 최신 정보를 실시간으로 참고할 때 유용합니다. 초급 개발자도 쉽게 활용할 수 있도록 라이브러리와 API가 많이 제공되고 있습니다.
------------------------------------------------------------
물론입니다! 백엔드 개발자의 관점에서 RAG(Retrieval-Augmented Generation)를 설명해드릴게요.

---

### RAG란 무엇인가?

RAG는 **Retrieval-Augmented Generation**의 약자로, 기존의 생성형 AI(예: GPT) 모델에 **정보 검색(Retrieval)** 기능을 결합한 기술입니다. 즉, 모델이 단순히 학습된 데이터만으로 답변을 생성하는 것이 아니라, 외부 데이터베이스나 문서 저장소에서 관련 정보를 먼저 검색한 뒤, 그 정보를 바탕으로 더 정확하고 구체적인 답변을 생성합니다.

---

### 백엔드 개발자의 관점에서 RAG 구성 요소

1. **문서 저장소 (Knowledge Base)**
   - 대량의 텍스트 데이터(예: 위키피디아, 사내 문서, 제품 매뉴얼 등)를 저장하는 곳입니다.
   - 보통 데이터베이스, 파일 스토리지, 혹은 벡터 데이터베이스(예: Pinecone, FAISS)로 구현합니다.

2. **검색 엔진 (Retriever)**
   - 사용자의 쿼리와 관련

,model,temperature,prompt,output,elapsed_sec,input_tokens,output_tokens,total_tokens,estimate_cost_usd
0,gpt-4.1-mini,0.2,RAG를 한 문장으로 설명해줘.,RAG( Retrieval-Augmented Generation )는 외부 지식베이...,2.921,29,40,None,0.000076
1,gpt-4.1-mini,0.2,RAG를 초급 개발자에게 5문장으로 설명해줘,"RAG는 ""Retrieval-Augmented Generation""의 약자입니다. ...",3.441,34,112,None,0.000193
2,gpt-4.1-mini,0.2,RAG를 백엔드 개발자의 관점에서 설명해줘,물론입니다! 백엔드 개발자의 관점에서 RAG(Retrieval-Augmented G...,11.423,32,844,None,0.001363
